In [ ]:
# !pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 13.1 MB/s eta 0:00:00


In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import urllib.request
from konlpy.tag import Okt
from tqdm import tqdm
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="ratings_train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt", filename="ratings_test.txt")

('ratings_test.txt', <http.client.HTTPMessage at 0x797dfc534210>)

In [ ]:
train_data = pd.read_csv('ratings_train.txt', sep='\t')
test_data = pd.read_csv('ratings_test.txt', sep='\t')

print(train_data.head())
print(test_data.head())

         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1
        id                                           document  label
0  6270596                                                굳 ㅋ      1
1  9274899                               GDNTOPCLASSINTHECLUB      0
2  8544678             뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아      0
3  6825595                   지루하지는 않은데 완전 막장임... 돈주고 보기에는....      0
4  6723715  3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??      0


In [ ]:
train_data.info()
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [ ]:
from konlpy.tag import Hannanum
hannanum = Hannanum()

train_data.document[1]

'흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나'

In [ ]:
hannanum.morphs(train_data.document[1])

['흠', '.', '..', '포스터보고', '초딩영화줄', '....', '오버연기', '조차', '가볍', '지', '않', '구나']

In [ ]:
def tokenize_text(text):
  word = re.sub(r"[^\sㄱ-ㅎㅏ-ㅣ가-힣]", " ", str(text))
  word_token = hannanum.morphs(word)
  return " ".join(word_token)

tokenize_text(train_data.document[1])

'흠 포스터보고 초딩영화줄 오버연기 조차 가볍 지 않 구나'

In [ ]:
stopwords = []

with open("Kor_StopWords.txt", "r", encoding="utf-8") as f:
    for line in f:
        stopwords.extend(line.strip().split(','))

print('한국어 stop words 갯수:', len(stopwords))
print(stopwords[:20])

한국어 stop words 갯수: 251
['아', ' 휴', ' 아이구', ' 아이쿠', ' 아이고', ' 어', ' 나', ' 우리', ' 저희', ' 따라', ' 의해', ' 을', ' 를', ' 에', ' 의', ' 가', ' 으로', ' 로', ' 에게', ' 뿐이다']


In [ ]:
train_data["document"] = train_data["document"].apply(tokenize_text)
test_data["document"] = test_data["document"].apply(tokenize_text)